# Source
https://vkvideo.ru/video-145052891_456248475

In [1]:
import warnings

In [2]:
warnings.simplefilter(action="ignore")

In [3]:
import pickle
import json
import random

import numpy as np

from lightfm import LightFM
from lightfm.data import Dataset

from replay.metrics import MAP, MRR, RocAuc, HitRate, NDCG, Coverage, Experiment
from replay.splitters import RandomSplitter

In [4]:
SEED = 23

In [5]:
random.seed(SEED)
np.random.seed(SEED)

In [6]:
USER_COL = "user_name"
ITEM_COL = "item_title"
RATING_COL = "rating"

In [7]:
MODEL = "google/gemini-2.5-flash"

In [8]:
output_dir_name = MODEL.split("/")[1]

In [ ]:
with open(f"gemini-2.5-flash/artificial_profiles_scores.pkl", "rb") as f:
    interactions = pickle.load(f)

with open(f"gemini-2.5-flash/artificial_profiles.json", "r", encoding="utf-8") as f:
    features_by_user = json.load(f)

with open(f"gemma-4-E2B-it/titles_with_tags_dict.pkl", "rb") as f:
    features_by_item = pickle.load(f)

print("Data loaded successfully:")
print(f"  - Users: {len(interactions)}")
print(f"  - Items: {len(features_by_item)}")
print(
    f"  - Total interactions: {sum(len(ratings) for ratings in interactions.values())}"
)

Data loaded successfully:
  - Users: 31
  - Items: 1147
  - Total interactions: 2697


In [10]:
sample_user = list(features_by_user.keys())[0]
print(f"Sample user: {sample_user}")
print(f"Bio: {features_by_user[sample_user]['bio'][:200]}...")
print(f"Tags: {features_by_user[sample_user]['tags']}")

Sample user: cultural_studies_enthusiast
Bio: A user interested in various aspects of culture, history, and social phenomena, with a particular focus on non-Western cultures, literature, and media....
Tags: ['arts_and_culture', 'middle_eastern_studies', 'asian_studies', 'folklore', 'postcolonialism']


In [11]:
user_to_test = random.choice(list(features_by_user.keys()))

In [12]:
user_to_test

'urban_cultural_researcher'

In [13]:
sample_item = list(features_by_item.keys())[0]
print(f"Sample item: {sample_item[:80]}...")
print(f"Tags: {features_by_item[sample_item]}")

Sample item: Исследование приоритетов и механизмов реализации отраслевых (секторальных) полит...
Tags: ['BRICS', 'development_studies', 'economics', 'politics', 'russian_studies']


In [14]:
import numpy as np
import pandas as pd

In [15]:
data = []

for user in interactions:
    for payload in interactions[user]:
        if interactions[user][payload] > 3:
            data.append(
                (user, json.loads(payload).get("title"), interactions[user][payload])
            )

df = pd.DataFrame(data=data, columns=["user_name", "item_title", "rating"])

In [16]:
df.shape

(2305, 3)

In [17]:
df.head()

,user_name,item_title,rating
0,cultural_studies_enthusiast,Запуск регулярного подкаста НИУ ВШЭ о странах ...,5
1,cultural_studies_enthusiast,История турецкой литературы и фольклора,5
2,cultural_studies_enthusiast,Китайский клуб НИУ ВШЭ 2025/2026,4
3,cultural_studies_enthusiast,Работа с материалами фольклорных экспедиций 2025,5
4,cultural_studies_enthusiast,Современное Искусство Ближнего Востока,5


In [18]:
data = []

for user in features_by_user:
    data.append((user, features_by_user[user]["tags"]))

df_users = pd.DataFrame(data=data, columns=["user_name", "features"])

In [19]:
df_users.head()

,user_name,features
0,cultural_studies_enthusiast,"[arts_and_culture, middle_eastern_studies, asi..."
1,educational_content_developer,"[education, teaching_methods, e-learning, cont..."
2,media_content_creator,"[content_marketing, smm, digital_media, promot..."
3,geopolitical_analyst_brics_middle_east,"[brics, global_politics, middle_eastern_studie..."
4,cultural_exchange_coordinator,"[cultural_exchange, arts_and_culture, event_ma..."


In [20]:
data = []

for item in features_by_item:
    data.append((item, features_by_item[item]))

df_items = pd.DataFrame(data=data, columns=["item_title", "features"])

In [21]:
df_items.head()

,item_title,features
0,Исследование приоритетов и механизмов реализац...,"[BRICS, development_studies, economics, politi..."
1,Антрополе - научно-популярный видео-подкаст о ...,"[digital_media, humanities, podcasting, sociol..."
2,"Разработка, создание и ведение сайта, посвящен...","[arts, culture, data_science, digital_media, h..."
3,Перевод с английского языка коллективной моног...,"[criminology, humanities, translation]"
4,Сеть военно-политических союзов в Евразии: баз...,"[data_science, eurasian_studies, politics, rus..."


In [22]:
dfg = df.groupby(by=ITEM_COL).agg({USER_COL: "nunique"}).reset_index()
dfg.columns = ["item_title", "user_name_count"]
dfg.sort_values(by="user_name_count", ascending=False)

,item_title,user_name_count
16,Digital Days на Факультете гуманитарных наук: ...,10
800,Цифровизация и отдача от образования в России,10
250,Китайский клуб НИУ ВШЭ 2025/2026,9
386,Особенности экономического сотрудничества Паки...,8
354,ОРЗ Изучении практик многоязычного обучения в ...,8
...,...,...
753,Теоретико-игровой подход к выбору брачного пар...,1
754,Теоретические основания музыкально-терапевтиче...,1
763,Трансформация доказательственной деятельности ...,1
98,Африканский семинар,1


In [23]:
df[USER_COL].nunique()

31

In [24]:
df[ITEM_COL].nunique()

850

In [25]:
train_dfs = []
test_dfs = []

for user_name in sorted(df.user_name.unique()):
    user_train, user_test = RandomSplitter(
        query_column=USER_COL,
        item_column=ITEM_COL,
        test_size=0.2,
    ).split(df[df.user_name == user_name])
    train_dfs.append(user_train)
    test_dfs.append(user_test)

train = pd.concat(train_dfs)
test = pd.concat(test_dfs)

In [26]:
df.shape

(2305, 3)

In [27]:
train.shape

(1841, 3)

In [28]:
train.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,african_and_middle_eastern_studies_researcher,12
1,ai_in_education_and_law_enthusiast,150
2,ai_in_education_innovator,62
3,art_and_culture_enthusiast,16
4,business_strategist,60
5,cultural_cinema_enthusiast,18
6,cultural_exchange_coordinator,35
7,cultural_studies_enthusiast,22
8,data_management_specialist,23
9,east_asia_cultural_specialist,22


In [29]:
train.user_name.nunique()

31

In [30]:
test.shape

(464, 3)

In [31]:
test.user_name.nunique()

31

In [32]:
test.groupby(by=["user_name"]).agg({"item_title": "nunique"}).reset_index()

,user_name,item_title
0,african_and_middle_eastern_studies_researcher,3
1,ai_in_education_and_law_enthusiast,38
2,ai_in_education_innovator,16
3,art_and_culture_enthusiast,4
4,business_strategist,15
5,cultural_cinema_enthusiast,5
6,cultural_exchange_coordinator,9
7,cultural_studies_enthusiast,5
8,data_management_specialist,6
9,east_asia_cultural_specialist,6


In [33]:
ALL_USERS = train[USER_COL].unique().tolist()
ALL_ITEMS = train[ITEM_COL].unique().tolist()

users = df_users[df_users[USER_COL].isin(ALL_USERS)]
items = df_items[df_items[ITEM_COL].isin(ALL_ITEMS)]

In [34]:
dataset = Dataset()

In [35]:
dataset.fit_partial(ALL_USERS, ALL_ITEMS)

In [36]:
user_mapping = dataset.mapping()[0]
item_mapping = dataset.mapping()[2]

inv_user_mapping = {v: k for k, v in user_mapping.items()}
inv_item_mapping = {v: k for k, v in item_mapping.items()}

In [37]:
train_interactions, train_weights = dataset.build_interactions(
    train[[USER_COL, ITEM_COL, RATING_COL]].values
)

In [38]:
train_interactions

<COOrdinate sparse matrix of dtype 'int32'
	with 1841 stored elements and shape (31, 786)>

In [39]:
train_weights

<COOrdinate sparse matrix of dtype 'float32'
	with 1841 stored elements and shape (31, 786)>

In [40]:
class LightFM4Rec:
    def __init__(
        self,
        model,
        user_mapping,
        item_mapping,
        inv_user_mapping,
        inv_item_mapping,
        user_col,
        item_col,
        rating_col,
    ):
        self.model = model
        self.user_mapping = user_mapping
        self.item_mapping = item_mapping
        self.inv_user_mapping = inv_user_mapping
        self.inv_item_mapping = inv_item_mapping
        self.user_col = user_col
        self.item_col = item_col
        self.rating_col = rating_col

    def fit(
        self,
        rating_matrix,
        train_interactions,
        user_features=None,
        item_features=None,
        epochs=16,
    ):
        self.user_features = user_features
        self.item_features = item_features
        self.train_interactions = train_interactions
        self.model.fit(
            rating_matrix,
            user_features=self.user_features,
            item_features=self.item_features,
            epochs=epochs,
        )

    def _get_lfm_pred(self, user_id):
        pred = self.model.predict(
            user_ids=user_id,
            item_ids=self.item_ids,
            user_features=self.user_features,
            item_features=self.item_features,
        )
        return pred

    def predict(self, test, interaction_matrix, user_col, filter_seen=True, k=10):
        user_ids = test[self.user_col].map(self.user_mapping).unique()
        self.item_ids = list(self.item_mapping.values())

        pred = pd.DataFrame(user_ids, columns=[user_col])
        scores = np.vstack(pred[user_col].apply(self._get_lfm_pred).values)

        if filter_seen:
            scores += np.nan_to_num(interaction_matrix.todense()[user_ids] * -np.inf)

        ind_part = np.argpartition(scores, -k)[:, -k:].copy()
        scores_not_sorted = np.take_along_axis(scores, ind_part, axis=1)
        ind_sorted = np.argsort(scores_not_sorted, axis=1)
        scores_sorted = np.sort(scores_not_sorted, axis=1)
        indices = np.take_along_axis(ind_part, ind_sorted, axis=1)

        preds = pd.DataFrame(
            {
                self.user_col: user_ids,
                self.item_col: np.flip(indices, axis=1).tolist(),
                self.rating_col: np.flip(scores_sorted, axis=1).tolist(),
            }
        ).explode([self.item_col, self.rating_col])
        preds[self.user_col] = preds[self.user_col].map(self.inv_user_mapping)
        preds[self.item_col] = preds[self.item_col].map(self.inv_item_mapping)

        return preds

In [41]:
lfm = LightFM(
    random_state=SEED,
    loss="warp",
    no_components=16,
)

In [42]:
model = LightFM4Rec(
    model=lfm,
    user_mapping=user_mapping,
    item_mapping=item_mapping,
    inv_user_mapping=inv_user_mapping,
    inv_item_mapping=inv_item_mapping,
    user_col=USER_COL,
    item_col=ITEM_COL,
    rating_col=RATING_COL,
)

In [43]:
model.fit(train_weights, train_interactions)

In [44]:
preds_wo_features = model.predict(test, train_interactions, USER_COL)

In [45]:
preds_wo_features[preds_wo_features["user_name"] == user_to_test]

,user_name,item_title,rating
29,urban_cultural_researcher,Китайский клуб НИУ ВШЭ 2025/2026,0.137475
29,urban_cultural_researcher,Гид абитуриента факультета социальных наук: пр...,0.112641
29,urban_cultural_researcher,Индийский клуб НИУ ВШЭ – Санкт-Петербург,-0.021272
29,urban_cultural_researcher,Менторы для иностранных слушателей центра русс...,-0.052555
29,urban_cultural_researcher,Улучшение имиджа Санкт-Петербурга как туристич...,-0.219457
29,urban_cultural_researcher,Работа с материалами фольклорных экспедиций 2025,-0.238796
29,urban_cultural_researcher,Телеграм канал Департамента социологии НИУ ВШЭ...,-0.440162
29,urban_cultural_researcher,Организация конференции НУГ «(Не)официальные г...,-0.454017
29,urban_cultural_researcher,Спикер на киноклубе,-0.494128
29,urban_cultural_researcher,"Интервью с выпускниками ОП ""Филология""",-0.504059


In [46]:
user_tags = users["features"].explode().unique()

In [47]:
item_tags = items["features"].explode().unique()

In [48]:
dataset.fit_partial(user_features=user_tags, item_features=item_tags)

In [49]:
sparse_u_features = dataset.build_user_features(
    [[row.user_name, row.features] for row in users.reset_index().itertuples()]
)

In [50]:
sparse_i_features = dataset.build_item_features(
    [[row.item_title, row.features] for row in items.reset_index().itertuples()]
)

In [51]:
lfm = LightFM(
    random_state=SEED,
    loss="warp",
    no_components=16,
)

In [52]:
model = LightFM4Rec(
    model=lfm,
    user_mapping=user_mapping,
    item_mapping=item_mapping,
    inv_user_mapping=inv_user_mapping,
    inv_item_mapping=inv_item_mapping,
    user_col=USER_COL,
    item_col=ITEM_COL,
    rating_col=RATING_COL,
)

In [53]:
model.fit(train_weights, train_interactions, sparse_u_features, sparse_i_features)

In [54]:
preds_w_features = model.predict(test, train_interactions, USER_COL)

In [55]:
preds_w_features[preds_w_features["user_name"] == user_to_test]

,user_name,item_title,rating
29,urban_cultural_researcher,Улучшение имиджа Санкт-Петербурга как туристич...,-0.329595
29,urban_cultural_researcher,Спикер на киноклубе,-0.416634
29,urban_cultural_researcher,Гид абитуриента факультета социальных наук: пр...,-0.54324
29,urban_cultural_researcher,Разработка маркетинговой стратегии для государ...,-0.579825
29,urban_cultural_researcher,Разработка комплекса маркетинга для банкетной ...,-0.695657
29,urban_cultural_researcher,Формирование ценностного предложения работодат...,-0.708715
29,urban_cultural_researcher,Организация конференции НУГ «(Не)официальные г...,-0.76476
29,urban_cultural_researcher,Китайский клуб НИУ ВШЭ 2025/2026,-0.769882
29,urban_cultural_researcher,Оформление социальных сетей Школы коммуникаций,-0.777266
29,urban_cultural_researcher,Телеграм канал Департамента социологии НИУ ВШЭ...,-0.795079


In [56]:
K = [10]
metrics = Experiment(
    [
        MAP(K),
        MRR(K),
        RocAuc(10),
        NDCG(K),
        HitRate(K),
        Coverage(K),
    ],
    test,
    train,
    query_column=USER_COL,
    item_column=ITEM_COL,
    rating_column=RATING_COL,
)

In [57]:
metrics.add_result("LFM_wo_features", preds_wo_features)
metrics.add_result("LFM_w_features", preds_w_features)

In [58]:
metrics.results

,MAP@10,MRR@10,RocAuc@10,NDCG@10,HitRate@10,Coverage@10
LFM_wo_features,0.294682,0.520200,0.463347,0.397005,0.741935,0.307888
LFM_w_features,0.435810,0.716795,0.632492,0.539130,0.870968,0.304071


In [59]:
data = []

for user_name in sorted(test.user_name.unique()):
    data.append(
        (
            user_name,
            len(
                set(
                    preds_w_features[
                        preds_w_features["user_name"] == user_name
                    ].item_title.unique()
                ).intersection(test[test["user_name"] == user_name].item_title.unique())
            ),
            test[test["user_name"] == user_name].item_title.nunique(),
        )
    )

check = pd.DataFrame(data=data, columns=["user_name", "intersection", "all_test"])
check["share"] = check.apply(lambda x: x.intersection / x.all_test, axis=1)

In [60]:
check.sort_values(by=["share", "intersection"], ascending=[False, False]).reset_index(
    drop=True
)

,user_name,intersection,all_test,share
0,cultural_cinema_enthusiast,3,5,0.600000
1,student_life_coordinator,7,13,0.538462
2,marketing_and_media_strategist,9,18,0.500000
3,event_organizer_and_community_builder,10,21,0.476190
4,financial_analyst_macro_and_micro,8,18,0.444444
5,cultural_exchange_coordinator,4,9,0.444444
6,global_economic_analyst,8,19,0.421053
7,project_manager_and_event_organizer,10,24,0.416667
8,linguistics_and_cultural_studies_enthusiast,8,20,0.400000
9,cultural_studies_enthusiast,2,5,0.400000


In [61]:
with open(f"{output_dir_name}/lightfm_sber_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [62]:
with open(f"{output_dir_name}/preds_wo_features.pkl", "wb") as f:
    pickle.dump(preds_wo_features, f)

In [63]:
with open(f"{output_dir_name}/preds_w_features.pkl", "wb") as f:
    pickle.dump(preds_w_features, f)

In [64]:
list(features_by_user.keys())

['cultural_studies_enthusiast',
 'educational_content_developer',
 'media_content_creator',
 'geopolitical_analyst_brics_middle_east',
 'cultural_exchange_coordinator',
 'sociology_and_cultural_studies_enthusiast',
 'geopolitical_analyst',
 'data_management_specialist',
 'marketing_and_media_strategist',
 'global_economic_analyst',
 'linguistics_and_cultural_studies_enthusiast',
 'software_developer_for_educational_platforms',
 'business_strategist',
 'east_asia_cultural_specialist',
 'legal_innovation_and_education_enthusiast',
 'geopolitical_analyst_global_economy',
 'psychology_researcher_mental_health_and_wellbeing',
 'project_manager_and_event_organizer',
 'event_organizer_and_community_builder',
 'financial_analyst_macro_and_micro',
 'cultural_cinema_enthusiast',
 'ai_in_education_and_law_enthusiast',
 'art_and_culture_enthusiast',
 'linguistic_ai_developer',
 'web_platform_developer',
 'ai_in_education_innovator',
 'media_and_communications_specialist',
 'african_and_middle_east

In [65]:
preds_wo_features[preds_wo_features["user_name"] == "financial_analyst_macro_and_micro"]

,user_name,item_title,rating
12,financial_analyst_macro_and_micro,Нестандартная монетарная политика: особенности...,-0.048572
12,financial_analyst_macro_and_micro,Трансформация структуры экономики России и сис...,-0.262122
12,financial_analyst_macro_and_micro,Анализ изменения режимов экспортного контроля ...,-0.349406
12,financial_analyst_macro_and_micro,"Антиинфляционная политика: теория, инструменты...",-0.359793
12,financial_analyst_macro_and_micro,Роль природного газа и атомной энергетики в со...,-0.381369
12,financial_analyst_macro_and_micro,Количественный анализ международной торговли,-0.404287
12,financial_analyst_macro_and_micro,Создание CRM-системы для торгового дома на меж...,-0.462269
12,financial_analyst_macro_and_micro,Экономические последствия отмены прямых выборо...,-0.466782
12,financial_analyst_macro_and_micro,Экономические циклы в российской экономике: пр...,-0.4977
12,financial_analyst_macro_and_micro,Факторы и источники инфляции в России,-0.499678


In [66]:
preds_w_features[preds_w_features["user_name"] == "financial_analyst_macro_and_micro"]

,user_name,item_title,rating
12,financial_analyst_macro_and_micro,Нестандартная монетарная политика: особенности...,1.193624
12,financial_analyst_macro_and_micro,Количественный анализ международной торговли,0.892633
12,financial_analyst_macro_and_micro,Мониторинг макроэкономической и финансовой сит...,0.471874
12,financial_analyst_macro_and_micro,Оценка роли выпуска субфедеральных облигаций в...,0.296406
12,financial_analyst_macro_and_micro,Бизнес и предпринимательство в странах БРИКС+:...,0.107252
12,financial_analyst_macro_and_micro,Мировая и российская практики бюджетирования б...,-0.004655
12,financial_analyst_macro_and_micro,Экономические факторы региональной дифференциа...,-0.156985
12,financial_analyst_macro_and_micro,Экономические циклы в российской экономике: пр...,-0.185542
12,financial_analyst_macro_and_micro,Оценка несоответствия границ субъектов РФ «ест...,-0.233535
12,financial_analyst_macro_and_micro,Влияние мер контроля за движением капитала на ...,-0.254926


In [67]:
test[test["user_name"] == "financial_analyst_macro_and_micro"]

,user_name,item_title,rating
1487,financial_analyst_macro_and_micro,Количественный анализ международной торговли,5
1489,financial_analyst_macro_and_micro,Мониторинг макроэкономической и финансовой сит...,5
1496,financial_analyst_macro_and_micro,Трансформация структуры экономики России и сис...,4
1499,financial_analyst_macro_and_micro,Бизнес и предпринимательство в странах БРИКС+:...,5
1506,financial_analyst_macro_and_micro,Анализ изменения режимов экспортного контроля ...,4
1510,financial_analyst_macro_and_micro,Разработка системы индикаторов налоговых право...,4
1513,financial_analyst_macro_and_micro,Роль природного газа и атомной энергетики в со...,4
1524,financial_analyst_macro_and_micro,Волонтёрство на XIII ежегодной международной к...,5
1529,financial_analyst_macro_and_micro,Анализ рынка коммерческой недвижимости,4
1531,financial_analyst_macro_and_micro,Мировая и российская практики бюджетирования б...,5


In [68]:
train[train["user_name"] == "financial_analyst_macro_and_micro"]

,user_name,item_title,rating
1555,financial_analyst_macro_and_micro,Теория и практика валютной политики в России,5
1516,financial_analyst_macro_and_micro,Привлечение финансирования под студенческий пр...,4
1527,financial_analyst_macro_and_micro,Цифровизация и отдача от образования в России,4
1484,financial_analyst_macro_and_micro,Исследование приоритетов и механизмов реализац...,4
1551,financial_analyst_macro_and_micro,Особенности фискальной политики в России,4
...,...,...,...
1546,financial_analyst_macro_and_micro,Каналы денежной трансмиссии и их особенности в...,4
1532,financial_analyst_macro_and_micro,Конструирование индекса «защитных» российских ...,5
1505,financial_analyst_macro_and_micro,Современные методы инвестиционного анализа,5
1502,financial_analyst_macro_and_micro,Стратегии модернизации экономик государств Юго...,4
